In [0]:
# Importando bibliotecas necessárias
from pyspark.sql import SparkSession
from datetime import datetime, timedelta
from pyspark.sql.functions import regexp_replace, to_date

# Inicializando SparkSession 
spark = SparkSession.builder.getOrCreate()

### Entendendo os dados

In [0]:
# Lendo a camada bronze da tabela de agências
df_agencias = spark.table("workspace.default.agencias")

In [0]:
# Contagem de registros, visualizar dados e esquema
print(f"Total de registros: {df_agencias.count()}")
display(df_agencias)

# Verificando campos nulos e se houver algum contar por coluna 
for col in df_agencias.columns:
    null_count = df_agencias.filter(df_agencias[col].isNull()).count()
    if null_count > 0:
        print(f"Coluna '{col}': {null_count} nulos")

In [0]:
# Lendo a camada bronze da tabela de clientes
df_clientes = spark.table("workspace.default.bronze_clientes")

In [0]:
# Contagem de registros, visualizar dados e esquema
print(f"Total de registros: {df_clientes.count()}")
display(df_clientes)

# Verificando campos nulos e se houver algum contar por coluna 
for col in df_clientes.columns:
    null_count = df_clientes.filter(df_clientes[col].isNull()).count()
    if null_count > 0:
        print(f"Coluna '{col}': {null_count} nulos")

In [0]:
# Remover prefixo '+55' dos telefones quando existir
df_clientes = df_clientes.withColumn(
    "telefone",
    regexp_replace("telefone", r"^\+55\s*", "")
)

# Transformar colunas de data em tipo date
df_clientes = df_clientes.withColumn("data_cadastro", to_date("data_cadastro", "yyyy-MM-dd"))
df_clientes = df_clientes.withColumn("vencimento", to_date("vencimento", "yyyy-MM-dd"))
df_clientes = df_clientes.withColumn("data_particao", to_date("data_particao", "yyyy-MM-dd"))

# Salvando a tabela Delta com os dados tratados (bronze para silver)
df_clientes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("data_particao") \
    .saveAsTable("workspace.default.silver_clientes")

display(df_clientes)

In [0]:
# Lendo a camada bronze da tabela de contratos
df_contratos = spark.table("workspace.default.bronze_contratos")

In [0]:
# Contagem de registros, visualizar dados e esquema
print(f"Total de registros: {df_contratos.count()}")
display(df_contratos)

# Verificar campos nulos e se houver algum contar por coluna 
for col in df_contratos.columns:
    null_count = df_contratos.filter(df_contratos[col].isNull()).count()
    if null_count > 0:
        print(f"Coluna '{col}': {null_count} nulos")

In [0]:
# Transformar colunas de data em tipo date
df_contratos = df_contratos.withColumn("data_contrato", to_date("data_contrato", "yyyy-MM-dd"))
df_contratos = df_contratos.withColumn("data_inicio", to_date("data_inicio", "yyyy-MM-dd"))
df_contratos = df_contratos.withColumn("data_fim", to_date("data_fim", "yyyy-MM-dd"))
df_contratos = df_contratos.withColumn("data_particao", to_date("data_particao", "yyyy-MM-dd"))

# Salvando a tabela Delta com os dados tratados (bronze para silver)
df_contratos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("data_particao") \
    .saveAsTable("workspace.default.silver_contratos")

display(df_contratos)